# 会说话的主播：一张角色图 + 一段文稿 → 口播视频

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dawang-sakura/colab-talking-avatar/blob/main/talking_avatar.ipynb)

```
文稿 ──VoxCPM2 配音──▶ 语音 wav ──ComfyUI + InfiniteTalk 对嘴──▶ 无声视频 ──合成──▶ talking_avatar.mp4
```

全程在 Google Colab 的 **L4 GPU** 上运行，你的电脑不用安装任何东西。

**使用步骤**
1. 菜单「代码执行程序 → 更改运行时类型」（英文界面：Runtime → Change runtime type）选 **L4 GPU** → 保存。
2. 只改下面的 **① 设置** 格（不改也能直接跑示例）。
3. 「代码执行程序 → 全部运行」（英文界面：Runtime → Run all）。弹出「此笔记本不是由 Google 创作」时点「仍然运行」。
4. 跑完浏览器会自动下载 `talking_avatar.mp4`。
5. **用完务必「代码执行程序 → 断开连接并删除运行时」**（英文界面：Runtime → Disconnect and delete runtime），闲置也会扣计算单元。

> **使用规范**：只能使用你本人、已取得当事人同意、或 AI 生成的虚构角色的脸和声音；禁止冒充他人。Colab 明文禁止 creating deepfakes（[Colab FAQ](https://research.google.com/colaboratory/faq.html)）。

没用过 Colab？请看仓库里的《新手操作手册》：`docs/新手操作手册.md`。

In [ ]:
# ① 设置（只需要改这一格）

# ===== 文稿 =====
# 一行一句；每句最多 80 字，超过会自动切开（VoxCPM2 单次太长容易出现啸叫或音色漂移）
# 句首可以用括号写语气，例如：(轻松地)大家好。开启「极致克隆」（见下方）时不能加语气括号
SCRIPT = """
大家好，我是虚构的 AI 主播。
一张图加一段文稿，就能让我开口说话。
全程在 Colab 上运行，不用装任何东西。
"""

# ===== 声音 =====
VOICE_DESC = "年轻女性，声音清亮自然，语速中等，语气亲切平稳，像新闻主播在播报"   # 音色设计用的文字描述
PAUSE_SEC = 0.4      # 句与句之间的停顿（秒）
SEED = 42            # 随机种子：音色设计、逐句配音、对嘴生成都用它；对声音不满意可以换一个数字重跑

# ===== 角色图 =====
# 默认用仓库自带的示例图（AI 生成的虚构人物）
# 换成自己的图：上传到左侧「文件」面板（会放在 /content/），再改成 "/content/你的图.png"
IMAGE_PATH = "examples/host.png"
# 对嘴画面的英文描述，可按角色外观补充（例如发型、服装）
PROMPT = ("A person is talking to the camera, facing the camera, upper body, "
          "natural head movement, clear mouth movement.")

# ===== 显存 =====
LOW_VRAM = False     # 对嘴阶段爆显存（OutOfMemory）时改成 True 再重跑：更慢但更省显存

# ===== 声音克隆（默认关闭）=====
# 只能用你本人、或已取得当事人同意的录音。开启后 VOICE_DESC 不再使用
ENABLE_VOICE_CLONE = False
REF_WAV = "/content/my_voice.wav"   # 参考录音（上传到 /content/）；建议 5～15 秒、单人说话、没有背景音乐
REF_TEXT = ""                       # 参考录音的逐字稿：填了走「极致克隆」（更像本人，但不能加语气括号）；留空走「可控克隆」
I_HAVE_CONSENT = False              # 确认录音是你本人、或已取得当事人同意后，才改成 True

if ENABLE_VOICE_CLONE and I_HAVE_CONSENT is not True:
    raise PermissionError("已开启声音克隆，但 I_HAVE_CONSENT 不是 True。"
                          "只能用你本人或已取得当事人同意的录音；确认后把 I_HAVE_CONSENT 改成 True 再运行。")
print("设置已读取")

In [ ]:
# ② 环境检查：GPU、示例文件、输入文件
import os, subprocess, sys, time

T_START = time.time()
REPO_URL = "https://github.com/dawang-sakura/colab-talking-avatar"
REPO_DIR = "/content/colab-talking-avatar"
OUT_DIR = "/content/output"
os.makedirs(OUT_DIR, exist_ok=True)


def run(cmd, **kw):
    """运行外部命令；失败时打印最后 40 行输出并报错。"""
    r = subprocess.run(cmd, capture_output=True, text=True, **kw)
    if r.returncode != 0:
        tail = "\n".join((r.stdout + r.stderr).splitlines()[-40:])
        raise RuntimeError(f"命令失败（exit {r.returncode}）：{' '.join(map(str, cmd))}\n{tail}")
    return r


def gpu_used():
    return subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader"],
                          capture_output=True, text=True).stdout.strip()


# GPU：本项目只在 L4（约 22.5GB 显存）上实测过
smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
                     capture_output=True, text=True).stdout.strip()
if not smi:
    raise RuntimeError("没有检测到 GPU：请到「代码执行程序 → 更改运行时类型」选 L4 GPU，保存后重新全部运行")
GPU_NAME, GPU_MIB = [s.strip() for s in smi.splitlines()[0].rsplit(",", 1)]
GPU_MIB = int(GPU_MIB)
print(f"GPU：{GPU_NAME}，显存 {GPU_MIB / 1024:.1f} GiB")
if "L4" not in GPU_NAME or GPU_MIB < 20 * 1024:
    print("⚠ 警告：当前不是 L4，或显存不足 20GB。本项目只在 L4 上实测过，T4（15GB）未测试，"
          "很可能爆显存或非常慢。\n"
          "  请到「代码执行程序 → 更改运行时类型」改成 L4；"
          "浏览器登录了多个 Google 账号时，Colab 默认用第一个账号，该账号没有计算单元会被降成 T4。")

# 取得示例文件（examples/）
if not os.path.isdir(REPO_DIR):
    r = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], capture_output=True, text=True)
    if r.returncode != 0:
        print("⚠ 克隆仓库失败，拿不到 examples/ 示例文件；如果用的是自己上传的图，可以忽略。\n" + r.stderr[-500:])


def resolve(p):
    """相对路径依序在 /content 和仓库目录里找；绝对路径原样返回。"""
    if os.path.isabs(p):
        return p
    for base in ("/content", REPO_DIR):
        if os.path.exists(os.path.join(base, p)):
            return os.path.join(base, p)
    return os.path.join(REPO_DIR, p)


IMAGE_FILE = resolve(IMAGE_PATH)
if not os.path.exists(IMAGE_FILE):
    raise FileNotFoundError(f"找不到角色图 {IMAGE_FILE}：请确认 IMAGE_PATH，或先在左侧「文件」面板上传")
print("角色图：", IMAGE_FILE)

if ENABLE_VOICE_CLONE:
    if I_HAVE_CONSENT is not True:
        raise PermissionError("只能用你本人或已取得当事人同意的录音；确认后把 I_HAVE_CONSENT 改成 True")
    REF_FILE = resolve(REF_WAV)
    if not os.path.exists(REF_FILE):
        raise FileNotFoundError(f"找不到参考录音 {REF_FILE}：请先在左侧「文件」面板上传，并确认 REF_WAV")
    print("参考录音：", REF_FILE)
print("环境检查完成")

## ③ 配音（VoxCPM2）
- 默认：用 `VOICE_DESC` 做一次**音色设计**，存成参考音 → 每句用**可控克隆**生成（同一把声音，句首括号可加语气）。
- 开启声音克隆：用你的参考录音克隆。`REF_TEXT` 留空＝可控克隆（可加语气）；填了逐字稿＝极致克隆（更像本人，**不能**加语气括号）。
- 句子之间插入 `PAUSE_SEC` 秒停顿，接成一条音轨 `/content/output/voice.wav`。
- 跑完会删除模型并释放显存，再进入对嘴。生成耗时大约等于音频长度。

In [ ]:
# ③ 配音：安装 VoxCPM2 → 音色设计（或用参考录音）→ 逐句生成 → 接成一条音轨 → 释放显存
import gc, re

VOXCPM_PIP = "git+https://github.com/OpenBMB/VoxCPM.git@cce58a5b59303c9bd63d12f23afe6a49a4a80c59"  # PyPI 版的 generate() 没有 seed 参数，固定这个 commit
DESIGN_TEXT = "大家好，欢迎收看今天的节目，我们马上开始。"   # 音色设计时念的参考句（只用来定声音，不会出现在成品里）
MAX_CHARS = 80
TONE_RE = re.compile(r"^\s*[(（]([^()（）]*)[)）]\s*")


def split_long(text, limit=MAX_CHARS):
    """超过 limit 字时优先在标点处切开，找不到标点就硬切。"""
    parts = []
    while len(text) > limit:
        cut = max(text.rfind(p, 0, limit) for p in "。！？；，、!?;,")
        cut = cut + 1 if cut > 0 else limit
        parts.append(text[:cut].strip())
        text = text[cut:].strip()
    if text:
        parts.append(text)
    return parts


# 解析文稿：[(语气 或 None, 句子)]；句首括号的语气套用到该行切出来的每一段
LINES = []
for raw in SCRIPT.strip().splitlines():
    raw = raw.strip()
    if not raw:
        continue
    m = TONE_RE.match(raw)
    tone = (m.group(1).strip() or None) if m else None
    body = raw[m.end():] if m else raw
    for part in split_long(body):
        LINES.append((tone, part))
if not LINES:
    raise ValueError("SCRIPT 是空的：请在 ① 设置格里填写文稿")
EXTREME = ENABLE_VOICE_CLONE and REF_TEXT.strip() != ""
if EXTREME and any(t for t, _ in LINES):
    raise ValueError("极致克隆（填了 REF_TEXT）不能和语气括号一起用：请去掉句首括号，或把 REF_TEXT 留空改走可控克隆")
print(f"共 {len(LINES)} 句：")
for i, (t, s) in enumerate(LINES, 1):
    print(f"  {i:02d} {'(' + t + ')' if t else ''}{s}")

t0 = time.time()
run([sys.executable, "-m", "pip", "install", "-q", VOXCPM_PIP])   # 会提示 gcsfs/fsspec 版本冲突，不影响
import numpy as np
import soundfile as sf
import torch
from voxcpm import VoxCPM

model = VoxCPM.from_pretrained("openbmb/VoxCPM2", load_denoiser=False)
SR = model.tts_model.sample_rate
print(f"VoxCPM2 就绪（采样率 {SR}），安装＋载入 {time.time() - t0:.0f} 秒")


def synth(text, **kw):
    return model.generate(text=text, cfg_value=2.0, inference_timesteps=10, seed=SEED, **kw)


if ENABLE_VOICE_CLONE:
    ref_path = f"{OUT_DIR}/ref_clone.wav"
    run(["ffmpeg", "-v", "error", "-y", "-i", REF_FILE, "-ac", "1", ref_path])   # 统一转成单声道 wav
    clone_kw = dict(reference_wav_path=ref_path)
    if EXTREME:
        # 极致克隆：prompt_wav_path + prompt_text（逐字稿必须和录音一字不差）
        clone_kw.update(prompt_wav_path=ref_path, prompt_text=REF_TEXT.strip())
    print("声音来源：", "极致克隆" if EXTREME else "可控克隆", REF_FILE)
else:
    # 音色设计：描述写在 text 开头的括号里；每次结果不同，所以只做一次并存成参考音，之后都用克隆
    wav = synth(f"({VOICE_DESC}){DESIGN_TEXT}")
    ref_path = f"{OUT_DIR}/voice_design_ref.wav"
    sf.write(ref_path, wav, SR)
    clone_kw = dict(reference_wav_path=ref_path)
    print(f"音色设计完成：{ref_path}（{len(wav) / SR:.2f} 秒）")

pause = np.zeros(int(PAUSE_SEC * SR), dtype=np.float32)
pieces = []
for i, (tone, text) in enumerate(LINES, 1):
    t1 = time.time()
    wav = np.asarray(synth(f"({tone}){text}" if tone else text, **clone_kw), dtype=np.float32).reshape(-1)
    if pieces:
        pieces.append(pause)
    pieces.append(wav)
    print(f"  {i:02d} 生成 {len(wav) / SR:.2f} 秒（耗时 {time.time() - t1:.0f} 秒）")
VOICE_WAV = f"{OUT_DIR}/voice.wav"
sf.write(VOICE_WAV, np.concatenate(pieces), SR)
T_VOICE = time.time() - t0
print(f"配音完成：{VOICE_WAV}，共 {sum(len(p) for p in pieces) / SR:.2f} 秒；本步耗时 {T_VOICE:.0f} 秒")

# 释放显存，留给对嘴阶段
del model
gc.collect()
torch.cuda.empty_cache()
print("已释放 VoxCPM2 显存，目前显存占用：", gpu_used())

from IPython.display import Audio, display
display(Audio(VOICE_WAV))

## ④ 对嘴（ComfyUI + InfiniteTalk）
- ComfyUI 固定 **v0.43.0**；模型 6 个文件全是 `.safetensors`，合计约 30GB，每次开机从 HuggingFace 下载（commit 已固定）。
- 画面 480×832（竖屏），6 步；角色图先裁成 480:832 比例（保留上方、裁掉下方）再缩放。
- 每段 81 帧（25fps，约 3.24 秒）；配音更长时一段接一段延伸（每段和前面重叠 9 帧当动作参考），最后裁回配音长度。
- 作者实测：21.5 秒的成品，这一步约 36 分钟。每 30 秒会打印一次进度。

In [ ]:
# ④-1 安装 ComfyUI（固定版本 v0.43.0）
COMFY = "/content/ComfyUI"
COMFY_TAG = "v0.43.0"   # 节点输入名称按这个版本的源码写，换版本可能要改节点图
t0 = time.time()
if not os.path.isdir(COMFY):
    run(["git", "clone", "--depth", "1", "--branch", COMFY_TAG, "https://github.com/comfyanonymous/ComfyUI", COMFY])
run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], cwd=COMFY)
T_INSTALL = time.time() - t0
print(f"ComfyUI {COMFY_TAG} 安装完成，{T_INSTALL:.0f} 秒")

In [ ]:
# ④-2 下载模型（只下载 .safetensors，约 30GB；来源与文件名照 ComfyUI 官方 InfiniteTalk 教程）
import shutil
from huggingface_hub import hf_hub_download

# (仓库, 文件路径, 固定的 commit, ComfyUI models 子目录)
MODELS = [
    ("Kijai/WanVideo_comfy_fp8_scaled", "I2V/Wan2_1-I2V-14B-480p_fp8_e4m3fn_scaled_KJ.safetensors",
     "033a4e487f60220b3d6e469599a6aebc46e13cee", "diffusion_models"),   # 16.6GB 底模 Wan2.1 I2V 14B 480P fp8
    ("Comfy-Org/Wan-2.1", "split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors",
     "123acf1cc74bccbb9bfff8ac1ee72edc08c2341d", "text_encoders"),      # 6.7GB 文本编码器
    ("Comfy-Org/Wan-2.1", "split_files/model_patches/wan2.1_infiniteTalk_single_fp16.safetensors",
     "123acf1cc74bccbb9bfff8ac1ee72edc08c2341d", "model_patches"),      # 5.1GB InfiniteTalk 单人
    ("Kijai/wav2vec2_safetensors", "wav2vec2-chinese-base_fp16.safetensors",
     "87847d3bc53702afda44078249e7c33e867827c4", "audio_encoders"),     # 0.19GB 中文 wav2vec2
    ("Kijai/WanVideo_comfy", "Wan2_1_VAE_bf16.safetensors",
     "8260d429d19fd7a72304cad059160b95d843913f", "vae"),                # 0.25GB
    ("Kijai/WanVideo_comfy", "Lightx2v/lightx2v_I2V_14B_480p_cfg_step_distill_rank64_bf16.safetensors",
     "8260d429d19fd7a72304cad059160b95d843913f", "loras"),              # 0.74GB 加速 LoRA
]
print(f"磁盘剩余 {shutil.disk_usage('/content').free / 1e9:.0f}GB（需要约 30GB）")
t0 = time.time()
for repo, path, rev, folder in MODELS:
    name = os.path.basename(path)
    assert name.endswith(".safetensors"), name
    dst = f"{COMFY}/models/{folder}/{name}"
    if os.path.exists(dst):
        continue
    print("下载：", name)
    tmp = hf_hub_download(repo, path, revision=rev, local_dir="/content/hf_tmp")
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.move(tmp, dst)
T_DOWNLOAD = time.time() - t0
print(f"模型就绪，{T_DOWNLOAD:.0f} 秒")

In [ ]:
# ④-3 准备输入：角色图裁成 480:832 并缩放；配音转成 16kHz 单声道给对嘴模型用
import math
from PIL import Image

WIDTH, HEIGHT = 480, 832   # 480P 竖屏；必须是 16 的倍数
STEPS = 6                  # 搭配 lightx2v 加速 LoRA，官方模板用 6 步
if not os.path.exists(VOICE_WAV):
    raise FileNotFoundError(f"找不到配音 {VOICE_WAV}：请先运行 ③")

IMG_NAME, AUD_NAME = "ta_host.png", "ta_voice.wav"
im = Image.open(IMAGE_FILE).convert("RGB")
w0, h0 = im.size
target = WIDTH / HEIGHT
if w0 / h0 > target:   # 太宽：左右各裁一半
    cw = round(h0 * target)
    box = ((w0 - cw) // 2, 0, (w0 - cw) // 2 + cw, h0)
else:                  # 太高：保留上方（头部），裁掉下方
    box = (0, 0, w0, round(w0 / target))
im.crop(box).resize((WIDTH, HEIGHT), Image.LANCZOS).save(f"{COMFY}/input/{IMG_NAME}")
print(f"图：{w0}x{h0} → 裁切 {box} → {WIDTH}x{HEIGHT}")

run(["ffmpeg", "-v", "error", "-y", "-i", VOICE_WAV, "-ac", "1", "-ar", "16000", f"{COMFY}/input/{AUD_NAME}"])
AUDIO_SEC = float(run(["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "csv=p=0",
                       f"{COMFY}/input/{AUD_NAME}"]).stdout)

# 每段 81 帧；超过就接延伸段（重叠 9 帧当动作参考），最后裁回配音长度
FPS, CHUNK, MOTION = 25, 81, 9
NEED_FRAMES = math.ceil(AUDIO_SEC * FPS)
N_CHUNKS = 1 + max(0, math.ceil((NEED_FRAMES - CHUNK) / (CHUNK - MOTION)))
print(f"配音 {AUDIO_SEC:.2f} 秒 → {NEED_FRAMES} 帧，分 {N_CHUNKS} 段生成")
if AUDIO_SEC > 22:
    print("⚠ 提示：作者实测过的最长成品是 21.5 秒（8 段，约 36 分钟）；更长时耗时按段数增加，系统内存够不够未验证")

In [ ]:
# ④-4 在后台启动 ComfyUI（只监听 127.0.0.1，不开网页界面）
import json, threading, urllib.request, urllib.error

API = "http://127.0.0.1:8188"
args = [sys.executable, "main.py", "--listen", "127.0.0.1", "--port", "8188"]
if LOW_VRAM:
    args.append("--novram")


def _serve():
    # 在后台线程里跑 ComfyUI，输出写进 /content/comfyui.log
    with open("/content/comfyui.log", "w") as log:
        subprocess.run(args, cwd=COMFY, stdout=log, stderr=subprocess.STDOUT)


def _post(path, payload):
    req = urllib.request.Request(API + path, data=json.dumps(payload).encode(),
                                 headers={"Content-Type": "application/json"})
    try:
        return json.load(urllib.request.urlopen(req))
    except urllib.error.HTTPError as e:
        raise RuntimeError(e.read().decode()) from None


def _get(path):
    return json.load(urllib.request.urlopen(API + path))


# 重跑时先关掉上一次留下的 ComfyUI（重启会话后它可能还占着显存和端口）
subprocess.run(["pkill", "-f", "main.py --listen 127.0.0.1 --port 8188"])
time.sleep(3)
t0 = time.time()
server = threading.Thread(target=_serve, daemon=True)
server.start()
for _ in range(300):
    if not server.is_alive():
        raise RuntimeError("ComfyUI 启动失败，请看 /content/comfyui.log")
    try:
        _get("/system_stats")
        break
    except (urllib.error.URLError, ConnectionError):
        time.sleep(1)
else:
    raise RuntimeError("等了 300 秒还没启动，请看 /content/comfyui.log")
T_BOOT = time.time() - t0
if "WanInfiniteTalkToVideo" not in _get("/object_info/WanInfiniteTalkToVideo"):
    raise RuntimeError("这个版本的 ComfyUI 没有 WanInfiniteTalkToVideo 节点")
print(f"ComfyUI 已启动，{T_BOOT:.0f} 秒；LOW_VRAM={LOW_VRAM}")

In [ ]:
# ④-5 生成对嘴视频（每 30 秒打印一次进度）
M = {folder: os.path.basename(path) for _, path, _, folder in MODELS}


def build_graph():
    # 节点与参数照官方模板 video_wan2_1_infinitetalk.json（单人）：lightx2v LoRA、euler、normal、cfg 1、audio_scale 1
    g = {}

    def add(cls, **inputs):
        nid = str(len(g) + 1)
        g[nid] = {"class_type": cls, "inputs": inputs}
        return nid

    unet = add("UNETLoader", unet_name=M["diffusion_models"], weight_dtype="default")
    model = add("LoraLoaderModelOnly", model=[unet, 0], lora_name=M["loras"], strength_model=1.0)
    clip = add("CLIPLoader", clip_name=M["text_encoders"], type="wan", device="default")
    vae = add("VAELoader", vae_name=M["vae"])
    patch = add("ModelPatchLoader", name=M["model_patches"])
    aenc = add("AudioEncoderLoader", audio_encoder_name=M["audio_encoders"])
    img = add("LoadImage", image=IMG_NAME)
    aud = add("LoadAudio", audio=AUD_NAME)
    aemb = add("AudioEncoderEncode", audio_encoder=[aenc, 0], audio=[aud, 0])
    pos = add("CLIPTextEncode", clip=[clip, 0], text=PROMPT)
    neg = add("ConditioningZeroOut", conditioning=[pos, 0])
    sampler = add("KSamplerSelect", sampler_name="euler")
    frames = None
    for i in range(N_CHUNKS):
        it_in = dict(mode="single_speaker", model=[model, 0], model_patch=[patch, 0], positive=[pos, 0],
                     negative=[neg, 0], vae=[vae, 0], width=WIDTH, height=HEIGHT, length=CHUNK,
                     start_image=[img, 0], audio_encoder_output_1=[aemb, 0], motion_frame_count=MOTION,
                     audio_scale=1.0)
        if frames is not None:
            it_in["previous_frames"] = [frames, 0]   # 延伸段：接在前面所有帧之后
        it = add("WanInfiniteTalkToVideo", **it_in)
        guider = add("CFGGuider", model=[it, 0], positive=[it, 1], negative=[it, 2], cfg=1.0)
        sigmas = add("BasicScheduler", model=[it, 0], scheduler="normal", steps=STEPS, denoise=1.0)
        noise = add("RandomNoise", noise_seed=SEED + i)
        lat = add("SamplerCustomAdvanced", noise=[noise, 0], guider=[guider, 0], sampler=[sampler, 0],
                  sigmas=[sigmas, 0], latent_image=[it, 3])
        dec = add("VAEDecode", samples=[lat, 0], vae=[vae, 0])
        if frames is None:
            frames = dec
        else:
            new = add("ImageFromBatch", image=[dec, 0], batch_index=MOTION, length=4096)   # 去掉重叠的动作参考帧
            frames = add("ImageBatch", image1=[frames, 0], image2=[new, 0])
    final = add("ImageFromBatch", image=[frames, 0], batch_index=0, length=NEED_FRAMES)
    video = add("CreateVideo", images=[final, 0], fps=float(FPS), audio=[aud, 0])
    save = add("SaveVideo", video=[video, 0], filename_prefix="talking_avatar/raw",
               format="auto", **{"format.codec": "auto"})
    return g, save


graph, save_id = build_graph()
t0 = time.time()
pid = _post("/prompt", {"prompt": graph})["prompt_id"]
last = 0
while True:
    hist = _get(f"/history/{pid}")
    if pid in hist:
        break
    if not server.is_alive():
        raise RuntimeError("ComfyUI 进程结束了（多半是系统内存或显存不够），请看 /content/comfyui.log")
    if time.time() - last >= 30:
        last = time.time()
        with open("/content/comfyui.log", "rb") as f:
            f.seek(max(0, os.path.getsize("/content/comfyui.log") - 400))
            tail = f.read().decode(errors="ignore").replace("\r", "\n").strip().splitlines()
        print(f"[{time.time() - t0:5.0f} 秒] {tail[-1][:150] if tail else ''}")
    time.sleep(2)
T_GEN = time.time() - t0

status = hist[pid].get("status", {})
if status.get("status_str") == "error":
    raise RuntimeError(json.dumps(status.get("messages"), ensure_ascii=False)[:3000])
v = hist[pid]["outputs"][save_id]["images"][0]
RAW_MP4 = os.path.join(COMFY, "output", v["subfolder"], v["filename"])
print(f"对嘴完成：{RAW_MP4}，生成耗时 {T_GEN / 60:.1f} 分钟")

In [ ]:
# ⑤ 合成输出：画面原样保留，音轨换回原始配音（ComfyUI 输出里的是转过的 16kHz 版）
from IPython.display import Video, display
from google.colab import files

OUT_MP4 = f"{OUT_DIR}/talking_avatar.mp4"
run(["ffmpeg", "-v", "error", "-y", "-i", RAW_MP4, "-i", VOICE_WAV, "-map", "0:v", "-map", "1:a",
     "-c:v", "copy", "-c:a", "aac", "-b:a", "192k", "-movflags", "+faststart", OUT_MP4])

print("=" * 60)
print(f"GPU：{GPU_NAME}")
print(f"成品：{OUT_MP4}（{WIDTH}x{HEIGHT}，配音 {AUDIO_SEC:.2f} 秒，{N_CHUNKS} 段）")
print(f"耗时：配音 {T_VOICE:.0f} 秒、装 ComfyUI {T_INSTALL:.0f} 秒、下载模型 {T_DOWNLOAD:.0f} 秒、"
      f"启动 {T_BOOT:.0f} 秒、对嘴 {T_GEN:.0f} 秒；从 ② 到现在共 {(time.time() - T_START) / 60:.1f} 分钟")
print("浏览器会自动下载；没弹出来就从左侧「文件」面板的 output 文件夹下载")
print("=" * 60)
files.download(OUT_MP4)
display(Video(OUT_MP4, embed=True, width=360))

## 跑完之后（重要）
- 确认浏览器下载文件夹里有 `talking_avatar.mp4`；没有就从左侧「文件」面板 `output/` 里下载。
- **「代码执行程序 → 断开连接并删除运行时」**（英文界面：Runtime → Disconnect and delete runtime）。**闲置也会扣计算单元。** 第一次点可能先弹出警告，确认后要再点一次删除。
- 爆显存（报错里有 `OutOfMemory` / `OOM`）：「代码执行程序 → 重启会话」→ ① 改 `LOW_VRAM = True` → 全部运行（已装的 ComfyUI 和已下载的模型会跳过）。